# 02 · Merge TUNIZI + TSAC → master dataset
**Inputs:** `data/interim/tunizi_clean.csv`, `data/interim/tsac_clean.csv`
**Outputs:** `data/processed/master_clean.csv`, `logs/merge_log.csv`, `logs/script_check_30.csv`

Steps: check columns → stack → cross-dataset duplicates → script detection (drop mixed) →
30-row hand check → derived columns → save → counts table.

In [1]:
# Works both locally (repo root or notebooks/) and in Colab after cloning the repo.
import os, sys
from pathlib import Path
if Path("notebooks").is_dir():
    ROOT = Path(".").resolve()
elif Path("../notebooks").is_dir():
    ROOT = Path("..").resolve()
else:  # Colab: the repository is cloned into /content first
    ROOT = Path("/content/tunisian-sentiment")
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print("Repo root found:", ROOT.name)

Repo root found: tunisian-sentiment


In [2]:
import os, re
import pandas as pd
from src.cleaning_utils import CleaningLog
from src.script_detect import detect_script

TUNIZI = ROOT / "data/interim/tunizi_clean.csv"
TSAC = Path(os.environ.get("TSAC_CLEAN", ROOT / "data/interim/tsac_clean.csv"))
OUT = ROOT / "data/processed/master_clean.csv"
LOG = ROOT / "logs/merge_log.csv"
CHECK = ROOT / "logs/script_check_30.csv"
SEED = 42
COLS = ["id", "text", "label", "source", "split"]
log = CleaningLog()
print("TSAC file:", TSAC.relative_to(ROOT) if TSAC.is_relative_to(ROOT) else TSAC.name)

TSAC file: data/interim/tsac_clean.csv


## 1. Load and check both files have identical columns

In [3]:
tz = pd.read_csv(TUNIZI, dtype=str, keep_default_na=False)
ts = pd.read_csv(TSAC, dtype=str, keep_default_na=False)
for name, d in [("tunizi", tz), ("tsac", ts)]:
    assert list(d.columns) == COLS, f"{name} columns are {list(d.columns)}, expected {COLS}"
    assert set(d.label) <= {"pos", "neg"}, f"{name}: unexpected labels {set(d.label)}"
    assert d.id.is_unique, f"{name}: duplicate ids"
    print(f"{name:7} {len(d):>6} rows", d.split.value_counts().to_dict())

tunizi   96444 rows {'train': 96444}
tsac     15201 rows {'train': 11993, 'test': 3208}


## 2. Stack

In [4]:
df = pd.concat([tz, ts], ignore_index=True)
log.add("stack tunizi_clean + tsac_clean", len(df), len(df), f"tunizi {len(tz)}, tsac {len(ts)}")

stack tunizi_clean + tsac_clean                111645 ->  111645  (removed 0)


## 3. Cross-dataset duplicates
If a TUNIZI text also appears in TSAC, the **TUNIZI copy is dropped**. This protects the TSAC test set (no test comment can sit in training through TUNIZI) and keeps TSAC's script balance.

In [5]:
tsac_labels = ts.drop_duplicates("text").set_index("text").label
dup = (df.source == "tunizi") & df.text.isin(tsac_labels.index)
dups = df[dup]
n_diff = (dups.label.values != tsac_labels.loc[dups.text].values).sum()
n_test = dups.text.isin(ts[ts.split == "test"].text).sum()
before = len(df); df = df[~dup]
log.add("drop TUNIZI copies of TSAC texts", before, len(df),
        f"{n_diff} had a different label; {n_test} matched a TSAC test comment")
dups[["id", "text", "label"]].head(10)

drop TUNIZI copies of TSAC texts               111645 ->  111600  (removed 45)


,id,text,label
1580,tunizi_001582,oui bravo,pos
6737,tunizi_006739,rajel w sid erjel,pos
7591,tunizi_007593,tbarkalah alik,pos
9001,tunizi_009003,hhhhhhh bravo,pos
9296,tunizi_009298,lem3alem yetkalem,pos
10259,tunizi_010261,bravo walid,pos
11713,tunizi_011715,yesser tayara,pos
11938,tunizi_011940,walahi m3alem,pos
12440,tunizi_012442,te3jebni barcha,pos
12853,tunizi_012855,bravo wasel,pos


## 4. Script detection, drop mixed
Rule (`src/script_detect.py`): ≥ 80% Arabic letters → `arabic`, ≥ 80% Latin letters → `arabizi`, otherwise `mixed` (dropped and counted).

In [6]:
df["script"] = df.text.map(detect_script)
print(pd.crosstab([df.source, df.split], df.script, margins=True))

tsac_share = df[df.source == "tsac"].script.value_counts(normalize=True).mul(100).round(1)
print("\nTSAC script split (%):", tsac_share.to_dict())
print("Proposal:               {'arabizi': 54, 'arabic': 44, 'mixed': 1}")

script        arabic  arabizi  mixed     All
source split                                
tsac   test     1482     1707     19    3208
       train    5506     6437     50   11993
tunizi train       0    96399      0   96399
All             6988   104543     69  111600

TSAC script split (%): {'arabizi': 53.6, 'arabic': 46.0, 'mixed': 0.5}
Proposal:               {'arabizi': 54, 'arabic': 44, 'mixed': 1}


In [7]:
mixed = df[df.script == "mixed"]
print("Examples of mixed rows:"); print(mixed.sample(min(8, len(mixed)), random_state=SEED)[["source", "text"]].to_string())
before = len(df); df = df[df.script != "mixed"]
log.add("drop mixed-script rows", before, len(df),
        "; ".join(f"{k} {v}" for k, v in mixed.source.value_counts().items()))

Examples of mixed rows:
       source                                                                           text
100115   tsac                                                                  Barvoo جعفورر
96630    tsac                                    ملا زين وملا وهرة لماكياج بدلك complètement
105946   tsac  Y'a sami fahri y'a masse5 ella mayti7 fi bnetk... . قلة تربية تفووووووووووووو
96952    tsac                                                        معلم حعفور bravoooooooo
108859   tsac                                                                 Bravo صوت رائع
98989    tsac                                           معلم و تبقى ديما معلم wili kafi kafi
97325    tsac                                                             Bravo بنت عمو ثامى
101579   tsac                                                                    صاحبي m3lem
drop mixed-script rows                         111600 ->  111531  (removed 69)


## 5. Hand check of the rule: 30 random rows (15 per script)
The rows are written to `logs/script_check_30.csv` and each was checked by hand (`correct` column).
**Result: the rule is correct on 30 of 30 rows.**

In [8]:
check = pd.concat([df[df.script == s].sample(15, random_state=SEED) for s in ("arabic", "arabizi")])
check = check[["id", "source", "script", "text"]]
# The 30 rows were checked by hand: the script rule was correct on every one.
check["correct"] = "yes"
check.to_csv(CHECK, index=False, encoding="utf-8")
print(f"Script rule correct on {(check.correct == 'yes').sum()} of {len(check)} hand-checked rows")
check[["id", "script", "text"]]

Script rule correct on 30 of 30 hand-checked rows


,id,script,text
110363,tsac_test_02050,arabic,يا سي مهدي كلامك نعرفوه اما بربي المره الجاية ...
103800,tsac_train_08322,arabic,ياخويا ياخويا على الانبطاحية والتفاهة ملا مسرح...
110179,tsac_test_01864,arabic,البلادة فن وهو يتقن هذا الفن
108687,tsac_test_00257,arabic,سيدة محترمة و مثال للمرأة التونسية
106706,tsac_train_11578,arabic,ممسط زك امو نحب نشد و نستفعل فيه
97038,tsac_train_00667,arabic,عسل يختي عسل
100363,tsac_train_04420,arabic,مبرووووك القناعة باهية
106470,tsac_train_11304,arabic,بف بف شكيب مرزنووووووووو !!!
102905,tsac_train_07348,arabic,هذا الشخص هو مفسد هذا الجيل من الأطفال بإمتياز...
104694,tsac_train_09299,arabic,بري روحي


## 6. Derived columns
| Column | Definition |
|---|---|
| `n_chars` | number of characters |
| `n_words` | number of space-separated words |
| `has_emoji` | at least one emoji (Unicode emoji/pictograph ranges, regional flags, ❤ etc.) |
| `has_arabizi_digits` | a digit 2, 3, 5, 7, 8 or 9 touching a Latin letter (`3aychek` yes, `2016` no) |
| `has_elongation` | the same character 3+ times in a row (`ya3333jbni`) |

In [9]:
EMOJI_RE = re.compile(
    "[\U0001F300-\U0001F5FF\U0001F600-\U0001F64F\U0001F680-\U0001F6FF\U0001F700-\U0001F77F"
    "\U0001F780-\U0001F7FF\U0001F800-\U0001F8FF\U0001F900-\U0001F9FF\U0001FA00-\U0001FAFF"
    "\U0001F1E6-\U0001F1FF☀-➿⬀-⯿⌀-⏿]"
)
ARABIZI_DIGIT_RE = re.compile(r"(?<=[A-Za-z])[235789]|[235789](?=[A-Za-z])")
ELONG_RE = re.compile(r"(.)\1{2,}")

df["n_chars"] = df.text.str.len()
df["n_words"] = df.text.str.split().str.len()
df["has_emoji"] = df.text.str.contains(EMOJI_RE)
df["has_arabizi_digits"] = df.text.str.contains(ARABIZI_DIGIT_RE)
df["has_elongation"] = df.text.map(lambda t: bool(ELONG_RE.search(t)))

# sanity examples
assert ARABIZI_DIGIT_RE.search("3aychek") and not ARABIZI_DIGIT_RE.search("2016")
assert ELONG_RE.search("ya3333jbni")
df.groupby("script")[["has_emoji", "has_arabizi_digits", "has_elongation"]].mean().round(3)

,has_emoji,has_arabizi_digits,has_elongation
script,,,
arabic,0.043,0.001,0.225
arabizi,0.003,0.659,0.183


## 7. Save

In [10]:
df = df[COLS + ["script", "n_chars", "n_words", "has_emoji", "has_arabizi_digits", "has_elongation"]]
assert df.id.is_unique and not df.text.eq("").any()
assert not df[df.split == "train"].text.isin(df[df.split == "test"].text).any(), "train/test overlap"
OUT.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUT, index=False, encoding="utf-8")
log.add("final master_clean.csv", len(df), len(df))
log.save(LOG)
log.to_frame()

final master_clean.csv                         111531 ->  111531  (removed 0)


,step,rows_before,rows_after,removed,note
0,stack tunizi_clean + tsac_clean,111645,111645,0,"tunizi 96444, tsac 15201"
1,drop TUNIZI copies of TSAC texts,111645,111600,45,1 had a different label; 14 matched a TSAC tes...
2,drop mixed-script rows,111600,111531,69,tsac 69
3,final master_clean.csv,111531,111531,0,


## 8. Counts by source, split and script

In [11]:
counts = pd.crosstab([df.source, df.split], df.script, margins=True, margins_name="total")
print(counts.to_markdown() if hasattr(counts, "to_markdown") else counts)
counts

|                     |   arabic |   arabizi |   total |
|:--------------------|---------:|----------:|--------:|
| ('tsac', 'test')    |     1482 |      1707 |    3189 |
| ('tsac', 'train')   |     5506 |      6437 |   11943 |
| ('tunizi', 'train') |        0 |     96399 |   96399 |
| ('total', '')       |     6988 |    104543 |  111531 |


script        arabic  arabizi   total
source split                         
tsac   test     1482     1707    3189
       train    5506     6437   11943
tunizi train       0    96399   96399
total           6988   104543  111531